# Ray-wise STEC Data Check

This notebook checks the processed ray-wise STEC dataset by plotting IPP longitude/latitude scatter maps colored by observed STEC. Each point is one receiver-satellite ray sample.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 7.5,
    "axes.labelsize": 8,
    "axes.titlesize": 8.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
})

cwd = Path.cwd().resolve()
if (cwd / "datasets" / "processed" / "raystec_china_20240505.parquet").exists():
    PROJECT_DIR = cwd
elif (cwd / "Topic1" / "datasets" / "processed" / "raystec_china_20240505.parquet").exists():
    PROJECT_DIR = cwd / "Topic1"
elif (cwd.parent / "datasets" / "processed" / "raystec_china_20240505.parquet").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError("Could not locate Topic1 processed dataset from the current working directory.")
DATA_PATH = PROJECT_DIR / "datasets" / "processed" / "raystec_china_20240505.parquet"
FIG_DIR = PROJECT_DIR / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## Plot Helpers

In [ ]:
def style_map_axis(ax, extent=(70, 150, -10, 70)):
    """Apply a restrained publication-style basemap."""
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_facecolor("#F7FAFC")
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="#F7FAFC", zorder=0)
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#ECECEC", edgecolor="none", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.45, edgecolor="#2F2F2F", zorder=1)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.35, edgecolor="#555555", linestyle=":", zorder=1)

    ax.set_xticks(np.arange(80, 151, 20), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(0, 71, 20), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f"))
    ax.tick_params(length=3, width=0.6, color="#333333", pad=2)
    ax.gridlines(
        xlocs=np.arange(80, 151, 20),
        ylocs=np.arange(0, 71, 20),
        linewidth=0.35,
        color="#9AA0A6",
        alpha=0.45,
        linestyle="--",
        draw_labels=False,
        zorder=1,
    )
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor("#222222")


def save_figure(fig, stem):
    fig.savefig(FIG_DIR / f"{stem}.png", dpi=450)
    fig.savefig(FIG_DIR / f"{stem}.svg")

## Load Data

In [ ]:
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

required_cols = ["timestamp", "ipp_lon", "ipp_lat", "stec_obs", "station", "prn", "elevation"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Stations: {df['station'].nunique()}")
print(f"PRNs: {df['prn'].nunique()}")
display(df[required_cols].head())

## Basic Quality Summary

In [ ]:
summary = pd.DataFrame({
    "missing": df[required_cols].isna().sum(),
    "min": df[required_cols].select_dtypes(include="number").min(),
    "max": df[required_cols].select_dtypes(include="number").max(),
})
display(summary)

display(df["stec_obs"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame())

## Single-epoch IPP Scatter Colored by STEC on a Map

A single epoch is easier to inspect for station/PRN matching problems, local spatial discontinuities, and strange coordinates. The basemap helps identify whether IPP locations are geographically reasonable.

In [ ]:
target_time = pd.Timestamp("2024-05-05 12:00:00")
epoch_df = df.loc[df["timestamp"] == target_time].dropna(subset=["ipp_lon", "ipp_lat", "stec_obs"])

if epoch_df.empty:
    nearest_time = df.loc[(df["timestamp"] - target_time).abs().idxmin(), "timestamp"]
    print(f"No samples at {target_time}; using nearest epoch {nearest_time}.")
    target_time = nearest_time
    epoch_df = df.loc[df["timestamp"] == target_time].dropna(subset=["ipp_lon", "ipp_lat", "stec_obs"])

cvals = epoch_df["stec_obs"].to_numpy()
vmin = np.nanpercentile(cvals, 2)
vmax = np.nanpercentile(cvals, 98)

fig = plt.figure(figsize=(7.0, 5.2), dpi=160)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
style_map_axis(ax)

sc = ax.scatter(
    epoch_df["ipp_lon"],
    epoch_df["ipp_lat"],
    c=cvals,
    cmap="viridis",
    s=10,
    alpha=0.82,
    vmin=vmin,
    vmax=vmax,
    transform=ccrs.PlateCarree(),
    zorder=2,
    linewidths=0,
    rasterized=False,
)

cb = fig.colorbar(sc, ax=ax, orientation="vertical", shrink=0.68, pad=0.018, aspect=28)
cb.set_label("Observed STEC (TECU)")
cb.outline.set_linewidth(0.6)
cb.ax.tick_params(length=2.5, width=0.6, labelsize=7)

ax.text(
    0.012,
    0.988,
    f"{target_time:%Y-%m-%d %H:%M UTC}\n"
    f"n = {len(epoch_df):,}; color: 2-98th pct.",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=7.2,
    linespacing=1.25,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 2.5},
    zorder=4,
)
ax.set_title("IPP distribution of ray-wise observed STEC", pad=7)

save_figure(fig, f"stec_ipp_scatter_{target_time:%Y%m%d_%H%M}")
plt.show()

display(epoch_df[["station", "prn", "ipp_lon", "ipp_lat", "elevation", "stec_obs"]].head(10))

## Optional: STEC Distribution

Use this histogram to check whether the color range is dominated by a few extreme values.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.4), constrained_layout=True)
ax.hist(df["stec_obs"].dropna(), bins=80, color="#4C78A8", alpha=0.9)
ax.axvline(vmin, color="black", linestyle="--", linewidth=1, label="1% / 99% color limits")
ax.axvline(vmax, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Observed STEC (TECU)")
ax.set_ylabel("Sample count")
ax.set_title("Observed STEC distribution")
ax.legend()
fig.savefig(FIG_DIR / "stec_obs_distribution.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "stec_obs_distribution.svg", bbox_inches="tight")
plt.show()